# Falcon Perception Multi-object Tracking

This tutorial demonstrates how to build an **open-vocabulary multi-object tracking** pipeline for video using **Falcon Perception** and an open-source tracker (**ByteTrack**). Given a video and a natural-language query (e.g. *"players in red"*), the pipeline detects or segments the described objects in every frame and links them across time with consistent track IDs, producing an annotated output video.

### Prerequisites
- GPU with CUDA available, preferably multi-GPU for faster inference with distributed batching.
- The `falcon_perception` package and its dependencies installed (`pip install -e .` from this repo).
- The `trackers` and `supervision` packages (`pip install trackers supervision`).
- Optional: the `ffmpeg` library for video compression.

### Steps
- Spawn multiple GPU workers and distribute batched inference across them for fast video processing.
- Falcon Perception runs in two modes: **detection** (bounding boxes) and **segmentation** (instance masks).
- Integrate a multi-object tracker (ByteTrack) so that each object receives a stable ID across frames.
- Visualize tracked results with color-coded boxes, masks, and motion-aware traces.

### Notebook structure
1. GPU worker setup
2. Batch inference function
3. Tracker & annotator setup
4. Frame annotation helpers
5. Video processing pipeline
6. Run detection and visualize
7. Run segmentation and visualize
8. Cleanup

In [ ]:
import sys
sys.path.insert(0, "~/Falcon-Perception") ## path to the repo

import threading
import time
from concurrent.futures import ThreadPoolExecutor
from queue import Queue

import cv2
import numpy as np
import torch
import torch.multiprocessing as mp
import supervision as sv
from decord import VideoReader
from IPython.display import Video
from tqdm.auto import tqdm

from falcon_perception import PERCEPTION_MODEL_ID, setup_torch_config
from trackers import ByteTrackTracker, MotionEstimator, MotionAwareTraceAnnotator
from gpu_worker_paged import run as gpu_worker_run

setup_torch_config()

Download / copy input videos for the demo into `inputs` sub-folder

In [ ]:
import os
NB_PATH = os.path.abspath('')
!mkdir -p {NB_PATH}/inputs
!mkdir -p {NB_PATH}/outputs

### 1. Spawn GPU Workers

Launches one model replica per GPU. Re-running this cell safely tears down
any existing workers before spawning fresh ones.

In [ ]:
MODEL_CONFIG = {
    "hf_model_id": PERCEPTION_MODEL_ID,
    "dtype": "bfloat16",
    "compile": True,
    "min_dimension": 512,
    "max_dimension": 1024,
    "max_new_tokens": 2048,
}


def spawn_gpu_workers(config: dict) -> tuple[list, list, list]:
    """Spawn one model-worker process per GPU and wait until all are ready."""
    num_gpus = torch.cuda.device_count()
    print(f"Available GPUs: {num_gpus}")

    try:
        mp.set_start_method("spawn", force=True)
    except RuntimeError:
        pass

    workers, in_qs, out_qs = [], [], []
    for gpu_id in range(num_gpus):
        in_q, out_q = mp.Queue(), mp.Queue()
        p = mp.Process(
            target=gpu_worker_run,
            args=(gpu_id, in_q, out_q, config),
            daemon=True,
        )
        p.start()
        workers.append(p)
        in_qs.append(in_q)
        out_qs.append(out_q)
        print(f"  Spawned worker for cuda:{gpu_id} (pid {p.pid})")

    for i in range(num_gpus):
        msg_type, gid = out_qs[i].get(timeout=600)
        assert msg_type == "ready"
        print(f"  cuda:{gid} model loaded ✓")

    print(f"\nGPUs: {num_gpus} — models loaded.")
    return workers, in_qs, out_qs


def shutdown_gpu_workers(workers, in_qs):
    """Gracefully shut down all GPU worker processes."""
    for q in in_qs:
        try:
            q.put_nowait(None)
        except Exception:
            pass
    for p in workers:
        if p.is_alive():
            p.terminate()
        p.join(timeout=5)
    print("All GPU workers shut down.")


# Tear down previous workers if re-running
if "gpu_workers" in globals():
    shutdown_gpu_workers(gpu_workers, _in_queues)

gpu_workers, _in_queues, _out_queues = spawn_gpu_workers(MODEL_CONFIG)
num_gpus = len(gpu_workers)

### 2. Batch Inference

Distributes frames evenly across GPU workers. Each worker runs Falcon
Perception and returns bounding boxes (+ masks when `task="segmentation"`).

In [ ]:
def falcon_detect_batch(
    frames_rgb: list[np.ndarray],
    queries: list[str],
    task: str,
) -> list[sv.Detections]:
    """Run Falcon Perception on a batch of frames across all GPU workers.

    Returns one sv.Detections per frame. When task="segmentation", each
    Detection includes per-object binary masks (.mask, shape N×H×W bool).
    """
    if not frames_rgb:
        return []
    chunk_size = max(1, (len(frames_rgb) + num_gpus - 1) // num_gpus)
    chunks = [
        frames_rgb[i : i + chunk_size]
        for i in range(0, len(frames_rgb), chunk_size)
    ]

    for gpu_idx, chunk in enumerate(chunks):
        _in_queues[gpu_idx].put((chunk, queries, task))

    all_dets: list[sv.Detections] = []
    for gpu_idx in range(len(chunks)):
        msg_type, xyxy_list, masks_list = _out_queues[gpu_idx].get(timeout=120)
        assert msg_type == "result"
        for frame_i, xyxy in enumerate(xyxy_list):
            n = len(xyxy)
            if n == 0:
                all_dets.append(sv.Detections.empty())
            else:
                mask = masks_list[frame_i]
                all_dets.append(sv.Detections(
                    xyxy=xyxy,
                    confidence=np.ones(n, dtype=np.float32),
                    class_id=np.zeros(n, dtype=int),
                    mask=mask.astype(bool) if mask is not None else None,
                ))
    return all_dets

### 3. Tracker & Annotators Factory

Returns a **fresh** tracker and annotator set each time it's called.
This avoids residual tracking state leaking between video runs
(since `tracker.reset()` doesn't fully clear internal state).

In [ ]:
def create_tracker_and_annotators():
    """Create a fresh tracker and full set of annotators (no residual state)."""
    palette = sv.ColorPalette.from_hex([
        "#ffff00", "#ff9b00", "#ff8080", "#ff66b2", "#ff66ff", "#b266ff",
        "#9999ff", "#3399ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00",
    ])

    trk = ByteTrackTracker(minimum_consecutive_frames=3)
    trk.reset()
    return dict(
        tracker          = trk,
        box_ann          = sv.BoxAnnotator(color=palette, color_lookup=sv.ColorLookup.TRACK),
        mask_ann         = sv.MaskAnnotator(color=palette, color_lookup=sv.ColorLookup.TRACK, opacity=0.35),
        label_ann        = sv.LabelAnnotator(color=palette, color_lookup=sv.ColorLookup.TRACK, text_color=sv.Color.WHITE, text_scale=0.5),
        det_box_ann      = sv.BoxAnnotator(color=palette, color_lookup=sv.ColorLookup.INDEX),
        det_mask_ann     = sv.MaskAnnotator(color=palette, color_lookup=sv.ColorLookup.INDEX, opacity=0.35),
        det_label_ann    = sv.LabelAnnotator(color=palette, color_lookup=sv.ColorLookup.INDEX, text_color=sv.Color.WHITE, text_scale=0.5),
        motion_trace_ann = MotionAwareTraceAnnotator(color=palette, color_lookup=sv.ColorLookup.TRACK, thickness=2, trace_length=100),
        motion_est       = MotionEstimator(max_points=500, min_distance=10, quality_level=0.01, ransac_reproj_threshold=1.0),
    )

### 4. Frame Annotation Helpers

Per-frame functions that annotate a BGR frame with detections.
Masks are drawn first (underneath), then boxes, traces, and labels on top.

In [ ]:
def annotate_tracked(frame_bgr, det, ann):
    """Track detections and annotate: mask → box → trace → label."""
    det = ann["tracker"].update(det)
    annotated = frame_bgr.copy()
    coord_transform = ann["motion_est"].update(frame_bgr)
    if det.mask is not None:
        annotated = ann["mask_ann"].annotate(annotated, det)
    else:
        annotated = ann["box_ann"].annotate(annotated, det)
    annotated = ann["motion_trace_ann"].annotate(
        annotated, det, coord_transform=coord_transform)
    labels = (
        [str(tid) for tid in det.tracker_id]
        if det.tracker_id is not None else []
    )
    return ann["label_ann"].annotate(annotated, det, labels)


### 5. Video Processing Pipeline

Three-stage **threaded pipeline** so GPU inference never waits on CPU
annotation or disk I/O:

```
inference (GPU) ──► annotation_q ──► annotate (CPU) ──► write_q ──► writer (I/O)
```

- **Stage 1 — Inference**: reads frame batches and distributes them across GPU workers.
- **Stage 2 — Annotation**: dequeues `(frames, detections)`, runs the tracker + draws overlays. When tracking is disabled, frames within a batch are annotated in parallel via a thread-pool.
- **Stage 3 — Writer**: dequeues annotated frames and writes them to the output MP4.

Bounded queues (`maxsize=2`) between stages limit memory to ~3 batches of frames
while keeping all stages busy.

In [ ]:
def _resize_longest_edge(frame: np.ndarray, max_size: int) -> np.ndarray:
    """Resize *frame* so its longest edge equals *max_size*, preserving aspect ratio."""
    h, w = frame.shape[:2]
    scale = max_size / max(h, w)
    if scale >= 1.0:
        return frame
    new_w, new_h = int(round(w * scale)), int(round(h * scale))
    return cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)


def process_video(
    source_path: str,
    target_path: str,
    task: str,
    queries: list[str],
    batch_size_per_gpu: int = 8,
    max_size: int | None = None,
):
    """End-to-end: read video → detect/segment → track → write annotated video.

    Three-stage threaded pipeline so GPU inference, CPU-side tracking /
    annotation, and video I/O all overlap:

        inference (GPU) ──► annotation_q ──► annotate (CPU) ──► write_q ──► writer (I/O)

    When use_tracker=False (stateless annotations), individual frames within
    a batch are annotated in parallel via a thread-pool.

    If *max_size* is set, annotated frames are resized so the longest edge
    equals *max_size* (aspect ratio preserved) before writing.  Annotation
    itself runs at full resolution so detection coordinates stay correct.

    Creates a fresh tracker and annotator set per call to avoid stale state.
    """
    ann = create_tracker_and_annotators()
    batch_size = batch_size_per_gpu * num_gpus

    print(f"Task    : {task}")
    print(f"Queries : {queries}")
    print(f"Tracker : {'ByteTrack'}")
    print(f"Batch   : {batch_size_per_gpu}/gpu × {num_gpus} gpus = {batch_size} frames")
    print(f"Source  : {source_path}")
    print(f"Target  : {target_path}")

    vr = VideoReader(source_path)
    total_frames = len(vr)
    fps = vr.get_avg_fps()
    h, w, _ = vr[0].shape

    if max_size is not None:
        scale = max_size / max(h, w)
        if scale < 1.0:
            out_w, out_h = int(round(w * scale)), int(round(h * scale))
        else:
            max_size = None
            out_w, out_h = w, h
        print(f"Output  : {out_w}×{out_h} (max_size={max_size})")
    else:
        out_w, out_h = w, h
        print(f"Output  : {out_w}×{out_h}")
        

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(target_path, fourcc, fps, (out_w, out_h))

    pbar = tqdm(total=total_frames, desc="Processing")

    SENTINEL = object()
    annotation_q: Queue = Queue(maxsize=2)
    write_q: Queue = Queue(maxsize=2)
    det_time_total = [0.0]
    track_time_total = [0.0]
    frame_count = [0]
    pipeline_error = [None]

    # ── Stage 1: read frames + GPU inference ────────────────────────
    def inference_stage():
        try:
            frame_buf: list[np.ndarray] = []
            for i in range(total_frames):
                frame_buf.append(vr[i].asnumpy())
                if len(frame_buf) == batch_size:
                    t0 = time.time()
                    dets = falcon_detect_batch(frame_buf, queries, task)
                    det_time_total[0] += time.time() - t0
                    annotation_q.put((frame_buf, dets))
                    frame_buf = []
            if frame_buf:
                t0 = time.time()
                dets = falcon_detect_batch(frame_buf, queries, task)
                det_time_total[0] += time.time() - t0
                annotation_q.put((frame_buf, dets))
        except Exception as exc:
            pipeline_error[0] = exc
        finally:
            annotation_q.put(SENTINEL)

    # ── Stage 2: tracking + annotation (CPU) ────────────────────────
    def annotation_stage():
        try:
            while True:
                t0 = time.time()
                item = annotation_q.get()
                if item is SENTINEL:
                    break
                frames_rgb, dets = item
                annotated_batch = [
                    annotate_tracked(fr[..., ::-1], det, ann)
                    for fr, det in zip(frames_rgb, dets)
                ]
                if max_size is not None:
                    annotated_batch = [
                        _resize_longest_edge(f, max_size) for f in annotated_batch
                    ]
                write_q.put(annotated_batch)
                track_time_total[0] += time.time() - t0
        except Exception as exc:
            pipeline_error[0] = exc
        finally:
            write_q.put(SENTINEL)

    # ── Stage 3: video writer (I/O) ──────────────────────────────
    def writer_stage():
        try:
            while True:
                item = write_q.get()
                if item is SENTINEL:
                    break
                for frame in item:
                    writer.write(frame)
                    frame_count[0] += 1
                pbar.update(len(item))
        except Exception as exc:
            pipeline_error[0] = exc

    threads = [
        threading.Thread(target=inference_stage, name="inference"),
        threading.Thread(target=annotation_stage, name="annotation"),
        threading.Thread(target=writer_stage, name="writer"),
    ]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    if pipeline_error[0] is not None:
        raise pipeline_error[0]

    ann["tracker"].reset()
    ann = None
    writer.release()
    pbar.close()
    n = frame_count[0]
    print(f"Model inference time: {det_time_total[0] / n:.3f}s/frame, {det_time_total[0]:.1f}s total")
    print(f"Tracker time: {track_time_total[0] / n:.3f}s/frame, {track_time_total[0]:.1f}s total")
    print(f"Done — {n} frames → {target_path}")
    return target_path

### 6a. Run — Detection

Set `TASK = "detection"` to get bounding-box-only results (faster, no masks).

In [ ]:
SOURCE_VIDEO  = f"{NB_PATH}/inputs/video1.mp4"          # path to the input video
TARGET_VIDEO  = f"{NB_PATH}/outputs/perception-det-track.mp4"
QUERIES       = ["players in red"]                      # multiple queries as list

process_video(
    source_path=SOURCE_VIDEO,
    target_path=TARGET_VIDEO,
    task="detection",
    queries=QUERIES,
    batch_size_per_gpu=2,
)

### 6b. Visualize Tracked Detection Outputs

In [ ]:
TARGET_VIDEO_COMPRESSED = TARGET_VIDEO.replace(".mp4", "-compressed.mp4")

!ffmpeg -y -loglevel error -i  "{TARGET_VIDEO}" -vcodec libx264 -crf 28 "{TARGET_VIDEO_COMPRESSED}"

Video(TARGET_VIDEO_COMPRESSED, embed=True, width=1080)

### 7a. Run — Segmentation

Set `TASK = "segmentation"` to get per-instance masks.
Tracked masks are colored consistently across frames by tracker ID.

In [ ]:
SOURCE_VIDEO  = f"{NB_PATH}/inputs/video2.mp4"        # path to the input video
TARGET_VIDEO  = f"{NB_PATH}/outputs/perception-seg-track.mp4"
QUERIES       = ["bike", "person"]                 # multiple queries as list

process_video(
    source_path=SOURCE_VIDEO,
    target_path=TARGET_VIDEO,
    task="segmentation",
    queries=QUERIES,
    batch_size_per_gpu=2,
)

### 7b. Visualize Tracked Segmentation Outputs

In [ ]:
TARGET_VIDEO_COMPRESSED = TARGET_VIDEO.replace(".mp4", "-compressed.mp4")

!ffmpeg -y -loglevel error -i  "{TARGET_VIDEO}" -vcodec libx264 -crf 28 "{TARGET_VIDEO_COMPRESSED}"

Video(TARGET_VIDEO_COMPRESSED, embed=True, width=1080)

### 8. Cleanup

Gracefully shut down all GPU worker processes.

In [ ]:
shutdown_gpu_workers(gpu_workers, _in_queues)